<a href="https://colab.research.google.com/github/mohiuddin2806-lang/machine-learning-/blob/main/7031_week_4_u2975262.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ▶ 1.1 Set up Hadoop 3.5.0 (one NameNode + one DataNode)
# Import paths, operating-system utilities, download and checksum tools.
from pathlib import Path
import os
import subprocess
import urllib.request
import urllib.error
import hashlib
import re
import tarfile
import time

# Import XML tools to create Hadoop and Hive configuration files.
from xml.etree import ElementTree as ET

# Import the HTTP client used to check the NameNode and DataNode web services.
import requests

# Set a persistent path within the current temporary Colab runtime.
LAB = Path('/content/cn7031-week4-hive')
# Create the working directory when it does not exist.
LAB.mkdir(parents=True, exist_ok=True)
# Specify the Hadoop and Java paths.
HADOOP = LAB / 'hadoop-3.5.0'
JAVA = Path('/usr/lib/jvm/java-17-openjdk-amd64')

# Run a system command and stop with a useful error if it fails.
def checked(args, timeout=600, env=None, cwd=None):
    # Capture standard output and errors as readable text.
    p = subprocess.run(args, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, timeout=timeout, env=env, cwd=cwd)
    # Check the exit status before using the command's result.
    if p.returncode:
        raise RuntimeError(p.stdout[-5000:])
    # Return the printed output to the caller.
    return p.stdout

# Match a digest's length (in hexadecimal characters) to the algorithm that produced it.
DIGEST_BY_LENGTH = {32: 'md5', 40: 'sha1', 64: 'sha256', 128: 'sha512'}

# Read the first checksum file a publisher provides next to a download.
def published_digest(url, suffixes):
    # Remember the filename so it can be removed from the checksum text.
    name = url.rsplit('/', 1)[-1]
    for suffix in suffixes:
        try:
            # Download the small checksum file, e.g. file.tar.gz.sha512.
            with urllib.request.urlopen(url + suffix, timeout=60) as response:
                text = response.read().decode(errors='replace')
        except (urllib.error.URLError, TimeoutError):
            # Try the next checksum format if this one is not published.
            continue
        # Remove the filename, then keep only the hexadecimal groups.
        tokens = re.split(r'[\s:=()*]+', text.replace(name, ' '))
        digest = ''.join(t for t in tokens if re.fullmatch(r'[0-9a-fA-F]+', t)).lower()
        # Return the algorithm and digest when the length is recognised.
        if len(digest) in DIGEST_BY_LENGTH:
            return DIGEST_BY_LENGTH[len(digest)], digest
    # No usable checksum file was found.
    return None, None

# Download a file, showing progress, and keep it only if it matches the published checksum.
def download_verified(url, dest, suffixes, required=True, checksum_url=None):
    # Download into a temporary name so a failed check leaves nothing behind.
    partial = dest.with_name(dest.name + '.part')
    # timeout=60: a connection that stalls for a minute raises an error instead of hanging forever.
    with urllib.request.urlopen(url, timeout=60) as response, partial.open('wb') as out:
        total = int(response.headers.get('Content-Length') or 0)
        done, step = 0, -1
        while chunk := response.read(1 << 20):
            out.write(chunk)
            done += len(chunk)
            # Report progress in 10% steps for large downloads only.
            if total > 50_000_000 and done * 10 // total > step:
                step = done * 10 // total
                print(f'  Download: {step * 10}%  ({done >> 20} of {total >> 20} MB)')
    # Look up the publisher's checksum for this exact file (a few attempts: the file is tiny but sites can be slow).
    for attempt in range(3):
        algorithm, expected = published_digest(checksum_url or url, suffixes)
        if algorithm:
            break
        time.sleep(3)
    if algorithm is None:
        # Stop, unless this download is allowed to proceed without a checksum.
        if required:
            partial.unlink()
            raise RuntimeError(f'No checksum published for {url}.')
        print(f'  Note: no checksum file found for {dest.name}; continuing without one.')
    else:
        # Calculate the digest of the downloaded file with the same algorithm.
        with partial.open('rb') as stream:
            actual = hashlib.file_digest(stream, algorithm).hexdigest()
        # Reject the file if the digests do not match.
        if actual != expected:
            partial.unlink()
            raise RuntimeError(f'Download checksum failed for {dest.name}.')
        print(f'  {algorithm.upper()} checksum verified: {dest.name}')
    # Give the verified file its final name.
    partial.replace(dest)

# Where to download Apache releases from, fastest first. The CDN and main site carry only the newest
# releases; a public mirror keeps older ones; the Apache archive keeps every release but can be slow.
APACHE_SITES = ('https://dlcdn.apache.org', 'https://downloads.apache.org',
                'https://repo.huaweicloud.com/apache', 'https://archive.apache.org/dist')
APACHE_ARCHIVE = 'https://archive.apache.org/dist'

# Download an Apache release, trying each site in turn (twice each) until one works.
def download_apache(path, dest, suffixes, required=True):
    problems = []
    for site in APACHE_SITES:
        for attempt in (1, 2):
            try:
                print(f'  Trying {site} …')
                # Checksums always come from Apache itself, even when the file comes from a mirror.
                official = site if 'apache.org' in site else APACHE_ARCHIVE
                download_verified(f'{site}/{path}', dest, suffixes, required, f'{official}/{path}')
                return
            except urllib.error.HTTPError as error:
                # Not on this site (e.g. 404 after a release moves to the archive): try the next site.
                problems.append(f'{site}: HTTP {error.code}')
                break
            except (OSError, RuntimeError) as error:
                # Timeout, dropped connection or bad checksum: wait, then retry.
                problems.append(f'{site}: {error}')
                print(f'  Problem: {error}. Retrying …')
                time.sleep(5)
    raise RuntimeError('Could not download ' + path + ' from any Apache site:\n' + '\n'.join(problems))

# Install Java 17 only when it is not already available.
if not (JAVA / 'bin/java').exists():
    # Update package metadata for the Colab VM.
    print('Installing Java …')
    checked(['apt-get', 'update', '-qq'])
    # Install the headless JDK required by Hadoop, Hive and Pig.
    checked(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jdk-headless'])

# Download, verify and unpack Hadoop only once per current VM.
if not (HADOOP / '.verified').exists():
    print('Downloading Hadoop from Apache (about 554 MB) …')
    archive = LAB / 'hadoop-3.5.0.tar.gz'
    download_apache('hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz', archive, ('.sha512',))
    # Extract the verified archive safely.
    print('Installing Hadoop …')
    with tarfile.open(archive) as bundle:
        bundle.extractall(LAB, filter='data')
    # Record successful installation and remove the compressed archive.
    (HADOOP / '.verified').write_text('ok')
    archive.unlink()

# Set Java and Hadoop environment variables for child processes (!, %%bash, hive, pig).
os.environ.update(JAVA_HOME=str(JAVA), HADOOP_HOME=str(HADOOP),
                  HADOOP_CONF_DIR=str(HADOOP / 'etc/hadoop'),
                  HADOOP_LOG_DIR=str(LAB / 'logs'), HADOOP_PID_DIR=str(LAB / 'pids'),
                  HADOOP_COMMON_HOME=str(HADOOP), HADOOP_MAPRED_HOME=str(HADOOP))
# Open the JDK internals that Hadoop, Hive and Pig still use on Java 17.
OPENS = ' '.join(f'--add-opens={m}=ALL-UNNAMED' for m in (
    'java.base/java.lang', 'java.base/java.lang.invoke', 'java.base/java.lang.reflect',
    'java.base/java.io', 'java.base/java.net', 'java.base/java.nio', 'java.base/java.util',
    'java.base/java.util.concurrent', 'java.base/java.util.concurrent.atomic',
    'java.base/java.util.regex', 'java.base/java.text', 'java.base/sun.nio.ch',
    'java.base/sun.nio.cs', 'java.base/sun.security.action', 'java.base/sun.util.calendar',
    'java.sql/java.sql'))
os.environ['HADOOP_CLIENT_OPTS'] = f'-Xmx1g {OPENS}'
# Put Hadoop and Java executables on the shell PATH.
os.environ['PATH'] = f'{HADOOP}/bin:{JAVA}/bin:' + os.environ['PATH']
# Make separate folders for namespace metadata, blocks, logs and process IDs.
for name in ('namenode', 'datanode', 'logs', 'pids'):
    (LAB / name).mkdir(exist_ok=True)

# Write a Hadoop-style XML configuration file (Hadoop's config folder unless another is given).
def config(name, values, folder=None):
    # Create the configuration root element.
    root = ET.Element('configuration')
    # Add each property name and its value.
    for key, value in values.items():
        property_node = ET.SubElement(root, 'property')
        ET.SubElement(property_node, 'name').text = key
        ET.SubElement(property_node, 'value').text = str(value)
    # Save the resulting configuration file.
    ET.ElementTree(root).write(Path(folder or HADOOP / 'etc/hadoop') / name, encoding='utf-8')

# Point HDFS clients at the local NameNode using port 9000.
config('core-site.xml', {
    'fs.defaultFS': 'hdfs://127.0.0.1:9000',
    'hadoop.tmp.dir': str(LAB / 'tmp'),
})
# Assign local storage directories and a default replication factor of one.
config('hdfs-site.xml', {
    'dfs.namenode.name.dir': (LAB / 'namenode').as_uri(),
    'dfs.datanode.data.dir': (LAB / 'datanode').as_uri(),
    'dfs.replication': 1,
    # Bind RPC and health-report endpoints to the VM's loopback address.
    'dfs.namenode.rpc-address': '127.0.0.1:9000',
    'dfs.namenode.rpc-bind-host': '127.0.0.1',
    'dfs.namenode.http-address': '127.0.0.1:9870',
    'dfs.namenode.http-bind-host': '127.0.0.1',
    # Keep DataNode transfer, HTTP and IPC endpoints on localhost.
    'dfs.datanode.address': '127.0.0.1:9866',
    'dfs.datanode.http.address': '127.0.0.1:9864',
    'dfs.datanode.ipc.address': '127.0.0.1:9867',
    'dfs.datanode.hostname': '127.0.0.1',
    'dfs.client.use.datanode.hostname': 'true',
    'dfs.datanode.use.datanode.hostname': 'true',
})
# Run Hive's MapReduce jobs in local mode (inside the HiveServer2 process, no YARN cluster).
config('mapred-site.xml', {
    'mapreduce.framework.name': 'local',
    # Keep each job's temporary staging files in one HDFS folder.
    'mapreduce.jobtracker.staging.root.dir': '/tmp/hadoop-staging',
})

# Add Java and a default heap size to Hadoop's environment script once.
# The heap can still be raised per process (HiveServer2 gets more in Step 1.2).
env = HADOOP / 'etc/hadoop/hadoop-env.sh'
if 'CN7031 environment' not in env.read_text():
    env.write_text(env.read_text() + f'\n# CN7031 environment\nexport JAVA_HOME={JAVA}\n'
                   'export HADOOP_HEAPSIZE_MAX=${HADOOP_HEAPSIZE_MAX:-512}\n')

# Show only warnings and errors from Hadoop's internals, so commands print short output.
log4j = HADOOP / 'etc/hadoop/log4j.properties'
QUIET = ['log4j.logger.org.apache.hadoop=WARN',
         'log4j.logger.org.apache.hadoop.fs.FileUtil=ERROR',
         'log4j.logger.org.apache.hadoop.mapred.LocalDistributedCacheManager=ERROR',
         'log4j.logger.org.apache.hadoop.metrics2.impl.MetricsSystemImpl=ERROR']
if log4j.exists():
    text = log4j.read_text()
    missing = [line for line in QUIET if line not in text]
    if missing:
        log4j.write_text(text + '\n# CN7031: quieter client output\n' + '\n'.join(missing) + '\n')

# Format only when no NameNode filesystem already exists.
if not (LAB / 'namenode/current/VERSION').exists():
    # Never silently overwrite an unexpectedly populated storage directory.
    assert all(not any((LAB / name).iterdir()) for name in ('namenode', 'datanode')), \
        'Existing storage found. Do not reformat it.'
    # Give the new HDFS namespace a fixed educational cluster ID.
    checked(['hdfs', 'namenode', '-format', '-nonInteractive', '-clusterid', 'CID-cn7031-week4'])

# Start a service only if its local HTTP endpoint is not responding.
for daemon, port in [('namenode', 9870), ('datanode', 9864)]:
    try:
        # Raise an error on a missing or unhealthy service endpoint.
        requests.get(f'http://127.0.0.1:{port}/jmx', timeout=2).raise_for_status()
    except requests.RequestException:
        # Start the selected Hadoop daemon in the background.
        checked(['hdfs', '--daemon', 'start', daemon])

# Wait until the NameNode reports exactly one live DataNode.
for attempt in range(45):
    try:
        # Query NameNode's JMX metrics over its local HTTP endpoint.
        response = requests.get('http://127.0.0.1:9870/jmx',
                                params={'qry': 'Hadoop:service=NameNode,name=FSNamesystemState'},
                                timeout=3)
        response.raise_for_status()
        # Exit the retry loop when the teaching cluster is ready.
        if response.json()['beans'][0]['NumLiveDataNodes'] == 1:
            break
    except (requests.RequestException, KeyError, IndexError):
        # Retry while the services finish their startup sequence.
        pass
    # Pause two seconds before asking again.
    time.sleep(2)
else:
    # Stop if the DataNode never becomes available.
    raise RuntimeError(f'Startup failed. Inspect the Hadoop logs in {LAB / "logs"}.')

# Wait until the NameNode has left safe mode, allowing writes.
checked(['hdfs', 'dfsadmin', '-safemode', 'wait'], timeout=90)
# Confirm that HDFS is usable before Hive is installed.
print('\nHDFS is ready ✓')
print('Hadoop 3.5.0 · NameNode running · 1 DataNode running')

Installing Java …
  Trying https://dlcdn.apache.org …
  Download: 0%  (1 of 554 MB)
  Download: 10%  (56 of 554 MB)
  Download: 20%  (111 of 554 MB)
  Download: 30%  (167 of 554 MB)
  Download: 40%  (222 of 554 MB)
  Download: 50%  (278 of 554 MB)
  Download: 60%  (333 of 554 MB)
  Download: 70%  (389 of 554 MB)
  Download: 80%  (444 of 554 MB)
  Download: 90%  (499 of 554 MB)
  Download: 100%  (554 of 554 MB)
  SHA512 checksum verified: hadoop-3.5.0.tar.gz
Installing Hadoop …

HDFS is ready ✓
Hadoop 3.5.0 · NameNode running · 1 DataNode running


In [ ]:
# @title ▶ 1.2 Install Apache Hive 4.1.0, start HiveServer2 and turn on %%hive cells
# Import process, network, text and notebook tools.
import os
import re
import shutil
import socket
import subprocess
import tarfile
import time
from pathlib import Path

import pexpect
from IPython.core.magic import register_cell_magic

# This cell needs the HDFS set-up from 1.1.
if 'LAB' not in globals() or 'download_apache' not in globals():
    raise RuntimeError('Run cell 1.1 first (in this runtime), then run this cell again.')

# Hive 4.1 is the release line that runs on Java 17, like this Hadoop.
HIVE_VERSION = '4.1.0'
HIVE = LAB / f'apache-hive-{HIVE_VERSION}-bin'
HIVE_CONF = HIVE / 'conf'

# Download, verify and unpack Hive only once per current VM.
if not (HIVE / '.verified').exists():
    print(f'Downloading Apache Hive {HIVE_VERSION} (about 470 MB) …')
    archive = LAB / 'hive.tar.gz'
    download_apache(f'hive/hive-{HIVE_VERSION}/apache-hive-{HIVE_VERSION}-bin.tar.gz', archive,
                    ('.sha256', '.sha512'))
    print('Installing Hive …')
    with tarfile.open(archive) as bundle:
        bundle.extractall(LAB, filter='data')
    for script in (HIVE / 'bin').iterdir():
        script.chmod(script.stat().st_mode | 0o111)
    (HIVE / '.verified').write_text('ok')
    archive.unlink()

# Hive and Hadoop each ship Google Guava; keep only Hadoop's newer copy so both use the same version.
hadoop_guava = sorted((HADOOP / 'share/hadoop/common/lib').glob('guava-*.jar'))
for old in (HIVE / 'lib').glob('guava-*.jar'):
    if hadoop_guava and old.name != hadoop_guava[-1].name:
        old.unlink()
        shutil.copy(hadoop_guava[-1], HIVE / 'lib')

# Folders Hive keeps on the local disk: metastore database, scratch space and logs.
for name in ('hive-local', 'hive-resources', 'hive-querylog', 'hive-oplogs'):
    (LAB / name).mkdir(exist_ok=True)

# Hive's configuration: an embedded Derby metastore, MapReduce in local mode, and HiveServer2 on port 10000.
config('hive-site.xml', {
    # The metastore: a small Derby database on the local disk that holds every table definition.
    'javax.jdo.option.ConnectionURL': f'jdbc:derby:;databaseName={LAB}/metastore_db;create=true',
    # Table data lives in HDFS under this folder.
    'hive.metastore.warehouse.dir': '/user/hive/warehouse',
    # Keep plain managed tables as they are (Hive 4 would otherwise translate them into external
    # tables stored elsewhere), so every table lives under /user/hive/warehouse/<database>.db/.
    'metastore.metadata.transformer.class': ' ',
    'metastore.warehouse.external.dir': '/user/hive/warehouse',
    'hive.metastore.warehouse.external.dir': '/user/hive/warehouse',
    # Run queries as MapReduce jobs (local mode, see mapred-site.xml).
    'hive.execution.engine': 'mr',
    'hive.exec.submitviachild': 'false',
    'hive.exec.submit.local.task.via.child': 'false',
    # Scratch and log folders.
    'hive.exec.scratchdir': '/tmp/hive',
    'hive.exec.local.scratchdir': str(LAB / 'hive-local'),
    'hive.downloaded.resources.dir': str(LAB / 'hive-resources'),
    'hive.querylog.location': str(LAB / 'hive-querylog'),
    'hive.server2.logging.operation.log.location': str(LAB / 'hive-oplogs'),
    # HiveServer2: listen on this machine only, and run queries as the connected user's process.
    'hive.server2.thrift.bind.host': '127.0.0.1',
    'hive.server2.thrift.port': 10000,
    'hive.server2.enable.doAs': 'false',
    # Do not send Hive's INFO/WARN progress log to beeline, so each cell shows only its result.
    'hive.server2.logging.operation.enabled': 'false',
    # Show column names without the table prefix (eid rather than employee.eid).
    'hive.resultset.use.unique.column.names': 'false',
    # Skip column statistics after every INSERT, which only slows a teaching cluster down.
    'hive.stats.column.autogather': 'false',
    'hive.metastore.event.db.notification.api.auth': 'false',
}, folder=HIVE_CONF)

# Environment for Hive's own programs (schematool, HiveServer2, beeline).
HIVE_ENV = {**os.environ, 'HIVE_HOME': str(HIVE), 'HIVE_CONF_DIR': str(HIVE_CONF), 'TERM': 'dumb',
            'HADOOP_HEAPSIZE_MAX': '2048', 'HADOOP_CLIENT_OPTS': f'-Xmx2g {OPENS}'}
os.environ.update(HIVE_HOME=str(HIVE), HIVE_CONF_DIR=str(HIVE_CONF))
os.environ['PATH'] = f'{HIVE}/bin:' + os.environ['PATH']

# HDFS folders Hive needs: scratch space and the warehouse.
checked(['hdfs', 'dfs', '-mkdir', '-p', '/tmp/hive', '/user/hive/warehouse'])
checked(['hdfs', 'dfs', '-chmod', '-R', '777', '/tmp'])

# Create the metastore database once. If an earlier attempt stopped half-way, start it again cleanly.
METASTORE_READY = LAB / '.metastore_ready'
if not METASTORE_READY.exists():
    print('Creating the Hive metastore …')
    shutil.rmtree(LAB / 'metastore_db', ignore_errors=True)
    checked([str(HIVE / 'bin/schematool'), '-dbType', 'derby', '-initSchema'],
            timeout=900, env=HIVE_ENV, cwd=LAB)
    METASTORE_READY.write_text('ok')

# Is HiveServer2 accepting connections on port 10000?
def hiveserver2_up():
    try:
        socket.create_connection(('127.0.0.1', 10000), timeout=2).close()
        return True
    except OSError:
        return False

# Start HiveServer2 in the background once, and wait until it accepts connections.
if not hiveserver2_up():
    print('Starting HiveServer2 (1–3 minutes) …')
    (LAB / 'logs').mkdir(exist_ok=True)
    log = open(LAB / 'logs/hiveserver2.out', 'ab')
    server = subprocess.Popen([str(HIVE / 'bin/hive'), '--service', 'hiveserver2'], stdout=log,
                              stderr=subprocess.STDOUT, env=HIVE_ENV, cwd=LAB, start_new_session=True)
    # Wait up to 5 minutes, and stop early if the server process exits with an error.
    for attempt in range(150):
        if hiveserver2_up() or server.poll() is not None:
            break
        time.sleep(2)
    if not hiveserver2_up():
        tail = (LAB / 'logs/hiveserver2.out').read_text(errors='replace').splitlines()[-40:]
        raise RuntimeError('HiveServer2 did not start. Last log lines:\n' + '\n'.join(tail))

# A long-running command-line shell (beeline or Pig's grunt) that notebook cells type into.
# State such as USE database or SET options is kept between cells, as in a terminal.
class ShellSession:
    def __init__(self, argv, prompt, run_command, echo_command, env, cwd):
        self.argv, self.prompt = argv, prompt
        self.run_command, self.echo_command = run_command, echo_command
        self.env, self.cwd, self.child, self.cells = env, cwd, None, 0

    # Start the shell and wait for its first prompt.
    def start(self, timeout=300):
        self.child = pexpect.spawn(self.argv[0], self.argv[1:], env=self.env, cwd=str(self.cwd),
                                   encoding='utf-8', codec_errors='replace', timeout=1800,
                                   dimensions=(1000, 1000))
        self.child.setecho(False)
        self.child.expect(self.prompt, timeout=timeout)
        return self.tidy(self.child.before)

    # Remove carriage returns, colour codes, bare prompts and the commands this class typed itself.
    def tidy(self, text, typed=()):
        text = re.sub(r'\x1b\[[0-9;?]*[A-Za-z]', '', text.replace('\r', ''))
        lines = []
        for line in text.split('\n'):
            if '__CN7031_DONE_' in line or any(t in line for t in typed):
                continue
            # Drop prompts at the start of a line (the shell may print several in a row).
            bare = re.sub(rf'^\s*(?:(?:{self.prompt})\s*)+', '', line)
            if bare != line and not bare.strip():
                continue
            lines.append(bare)
        return '\n'.join(lines).strip('\n')

    # Send one cell: save it as a script, run it in the shell, and wait for a unique end marker.
    def run(self, text):
        if self.child is None or not self.child.isalive():
            if self.child is not None:
                print('(The previous session had stopped; a new one was started. '
                      'Run your USE and SET statements again.)')
            self.start()
        self.cells += 1
        script = Path(self.cwd) / f'cell_{self.cells}.txt'
        script.write_text(text + '\n')
        marker = f'__CN7031_DONE_{self.cells}__'
        typed = (self.run_command.format(script), self.echo_command.format(marker))
        for command in typed:
            self.child.sendline(command)
        # The marker is printed by the shell's echo command (not the echoed command line itself).
        # If the cell closed the shell (quit), stop waiting and say how to open a new one.
        if self.child.expect([rf'(?<!echo ){marker}', pexpect.EOF]) == 1:
            output = self.tidy(self.child.before, typed)
            self.child.close()
            return (output + '\n' if output else '') + '(The shell has closed. The next cell will start a new one.)'
        output = self.child.before
        self.child.expect(self.prompt)
        return self.tidy(output, typed)

# Remove full-line -- comments and make sure the last statement ends with a semicolon.
def hiveql(cell):
    lines = [line for line in cell.strip().split('\n') if not line.strip().startswith(('--', '#'))]
    text = '\n'.join(lines).strip()
    return text if text.endswith(';') else text + ';'

# One beeline session connected to HiveServer2, kept open for the whole notebook.
HIVE_SESSION = ShellSession(
    [str(HIVE / 'bin/beeline'), '-u', 'jdbc:hive2://127.0.0.1:10000/default', '-n', 'root', '-p', '',
     '--color=false', '--showWarnings=false', '--outputformat=table'],
    prompt=r'\d+: jdbc:hive2://\S*> ', run_command='!run {}', echo_command='!sh echo {}',
    env=HIVE_ENV, cwd=LAB / 'hive-local')

# Notebook shortcut: a %%hive cell runs its HiveQL in the beeline session, like the Hue Hive editor.
# Keep only what matters in beeline's output: the result table, messages and errors, and the row count
# with its time. Hive's log lines and beeline's echo of the statement (already visible in the cell) are dropped.
def clean_hive_output(output, statement):
    typed = {l.strip().rstrip(';').strip() for l in statement.split('\n') if l.strip()}
    kept, in_echo = [], False
    for line in output.split('\n'):
        if line.startswith('>>>  '):
            in_echo = not line.rstrip().endswith(';')
            continue
        if in_echo:
            in_echo = not line.rstrip().endswith(';')
            continue
        if re.match(r'(INFO|WARN|DEBUG)\s+:', line) or line.strip().rstrip(';').strip() in typed:
            continue
        kept.append(line)
    return '\n'.join(kept).strip('\n')

@register_cell_magic
def hive(line, cell):
    text = hiveql(cell)
    if not text.strip(' ;\n'):
        print('(Nothing to run yet: type a HiveQL statement in this cell, ending with ;)')
        return
    print(clean_hive_output(HIVE_SESSION.run(text), text))

# Allow a comment line above %%hive or %%pig: move the %% line to the top, as Jupyter requires.
def comment_before_cell_magic(lines):
    first = next((k for k, l in enumerate(lines) if l.strip() and not l.lstrip().startswith('#')), None)
    if first and lines[first].startswith('%%'):
        return [lines[first]] + lines[:first] + lines[first + 1:]
    return lines

_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [f for f in _ip.input_transformers_cleanup
                                     if getattr(f, '__name__', '') != 'comment_before_cell_magic']
_ip.input_transformers_cleanup.append(comment_before_cell_magic)

# Connect beeline now, so the first %%hive cell answers quickly.
if HIVE_SESSION.child is None or not HIVE_SESSION.child.isalive():
    banner = HIVE_SESSION.start()
    print('\n'.join(l for l in banner.split('\n') if l.startswith(('Connected to', 'Driver'))))
print('\nHive is ready ✓  HiveServer2 on port 10000 · %%hive cells are ready ✓')

  Trying https://dlcdn.apache.org …
  Trying https://downloads.apache.org …
  Trying https://repo.huaweicloud.com/apache …
  Download: 0%  (1 of 469 MB)
  Download: 10%  (47 of 469 MB)
  Download: 20%  (94 of 469 MB)
  Download: 30%  (141 of 469 MB)
  Download: 40%  (188 of 469 MB)
  Download: 50%  (235 of 469 MB)
  Download: 60%  (282 of 469 MB)
  Download: 70%  (329 of 469 MB)
  Download: 80%  (376 of 469 MB)
  Download: 90%  (423 of 469 MB)
  Download: 100%  (469 of 469 MB)
  SHA256 checksum verified: hive.tar.gz
Installing Hive …
Creating the Hive metastore …
Starting HiveServer2 (1–3 minutes) …
Connected to: Apache Hive (version 4.1.0)
Driver: Hive JDBC (version 4.1.0)

Hive is ready ✓  HiveServer2 on port 10000 · %%hive cells are ready ✓


In [ ]:
# List the running Java processes: NameNode, DataNode and two RunJar.
!jps

4594 RunJar
3603 NameNode
3668 DataNode
4925 RunJar
5069 Jps


In [ ]:
# Ask HiveServer2 which version of Hive it is running.
%%hive
SELECT version();

+--------------------------------------------------+
|                       _c0                        |
+--------------------------------------------------+
| 4.1.0 r75e40b7537c91a70ccaa31c397d21823c7528eeb  |
+--------------------------------------------------+
1 row selected (6.03 seconds)


In [ ]:
# List the uploaded text files with their sizes.
!ls -l /content/ *.txt

-rw-r--r-- 1 root root  192 Oct 26 19:11 Customers.txt
-rw-r--r-- 1 root root  126 Oct 26 19:11 Orders.txt
-rw-r--r-- 1 root root  164 Oct 26 19:11 Sample.txt
-rw-r--r-- 1 root root  228 Oct 26 19:11 wordcount_simple.txt

/content/:
total 24
drwxr-xr-x 13 root root 4096 Oct 26 19:15 cn7031-week4-hive
-rw-r--r--  1 root root  192 Oct 26 19:11 Customers.txt
-rw-r--r--  1 root root  126 Oct 26 19:11 Orders.txt
drwxr-xr-x  1 root root 4096 Oct 21 13:23 sample_data
-rw-r--r--  1 root root  164 Oct 26 19:11 Sample.txt
-rw-r--r--  1 root root  228 Oct 26 19:11 wordcount_simple.txt


In [ ]:
# Report the type and line endings of each data file.
!file /content/*.txt

/content/Customers.txt:        ASCII text, with CRLF line terminators
/content/Orders.txt:           ASCII text, with CRLF line terminators
/content/Sample.txt:           ASCII text, with CRLF line terminators
/content/wordcount_simple.txt: ASCII text, with no line terminators


In [ ]:
# Show the first two lines of Sample.txt with tabs (^I) and carriage returns (^M) made visible.
!head -n 2 /content/Sample.txt | cat -A

1201^IGopal^I45000^ITechnical^Imanager^M$
1202^IManisha^I45000^IProof^Ireader^M$


In [ ]:
# Remove the Windows carriage returns from the three table files.
!sed -i 's/\r$//' /content/Sample.txt /content/Customers.txt /content/Orders.txt

In [ ]:
# Check the line endings again: CRLF should be gone.
!file /content/*.txt

/content/Customers.txt:        ASCII text
/content/Orders.txt:           ASCII text
/content/Sample.txt:           ASCII text
/content/wordcount_simple.txt: ASCII text, with no line terminators


In [ ]:
# Show the first two lines again: they should end in $ without ^M.
!head -n 2 /content/Sample.txt | cat -A

1201^IGopal^I45000^ITechnical^Imanager$
1202^IManisha^I45000^IProof^Ireader$


In [ ]:
# Create a database for your lab group (here group 6).
%%hive
CREATE SCHEMA userdb6;

No rows affected (0.408 seconds)


In [ ]:
# List the databases.
%%hive
SHOW DATABASES;

+----------------+
| database_name  |
+----------------+
| default        |
| userdb6        |
+----------------+
2 rows selected (0.183 seconds)


In [ ]:
# Make userdb6 the current database for the rest of the session.
%%hive
USE userdb6;

No rows affected (0.048 seconds)


In [ ]:
# Create an empty employee table for tab-separated text files.
%%hive
CREATE TABLE IF NOT EXISTS employee (eid int, name String, salary Float, designation String)
COMMENT 'Employee details'
ROW FORMAT DELIMITED FIELDS TERMINATED BY '\t' LINES TERMINATED BY '\n'
STORED AS TEXTFILE;

No rows affected (0.676 seconds)


In [ ]:
%%hive
LOAD DATA LOCAL INPATH '/content/Sample.txt' OVERWRITE INTO TABLE employee;

No rows affected (1.535 seconds)


In [ ]:
# Show the rows of the employee table.
%%hive
SELECT * FROM employee LIMIT 10;

+-------+--------------+----------+--------------+
|  eid  |     name     |  salary  | designation  |
+-------+--------------+----------+--------------+
| 1201  | Gopal        | 45000.0  | Technical    |
| 1202  | Manisha      | 45000.0  | Proof        |
| 1203  | Masthanvali  | 40000.0  | Technical    |
| 1204  | Kiran        | 40000.0  | Hr           |
| 1205  | Kranthi      | 30000.0  | Op           |
+-------+--------------+----------+--------------+
5 rows selected (0.472 seconds)


In [ ]:
# Show the full table definition; find the Location line.
%%hive
DESCRIBE FORMATTED employee;

+-------------------------------+----------------------------------------------------+----------------------------------------------------+
|           col_name            |                     data_type                      |                      comment                       |
+-------------------------------+----------------------------------------------------+----------------------------------------------------+
| eid                           | int                                                |                                                    |
| name                          | string                                             |                                                    |
| salary                        | float                                              |                                                    |
| designation                   | string                                             |                                                    |
|                   

In [ ]:
%%hive
LOAD DATA LOCAL INPATH '/content/Sample.txt' OVERWRITE INTO TABLE employee;

No rows affected (0.794 seconds)


In [ ]:
# List the files inside the employee table's HDFS folder.
!hdfs dfs -ls /user/hive/warehouse/userdb6.db/employee

Found 1 items
-rw-r--r--   1 root supergroup        160 2026-10-26 19:22 /user/hive/warehouse/userdb6.db/employee/Sample.txt


In [ ]:
# Print the table's data file: it is exactly the file you uploaded.
!hdfs dfs -cat /user/hive/warehouse/userdb6.db/employee/Sample.txt

1201	Gopal	45000	Technical	manager
1202	Manisha	45000	Proof	reader
1203	Masthanvali	40000	Technical	writer
1204	Kiran	40000	Hr	Admin
1205	Kranthi	30000	Op	Admin

In [ ]:
# Add a dept column to the table definition.
%%hive
ALTER TABLE employee ADD COLUMNS (dept STRING COMMENT 'Department name');

No rows affected (0.336 seconds)


In [ ]:
# Show the table again: the dept column is filled from the same file.
%%hive
SELECT * FROM employee;

+-------+--------------+----------+--------------+----------+
|  eid  |     name     |  salary  | designation  |   dept   |
+-------+--------------+----------+--------------+----------+
| 1201  | Gopal        | 45000.0  | Technical    | manager  |
| 1202  | Manisha      | 45000.0  | Proof        | reader   |
| 1203  | Masthanvali  | 40000.0  | Technical    | writer   |
| 1204  | Kiran        | 40000.0  | Hr           | Admin    |
| 1205  | Kranthi      | 30000.0  | Op           | Admin    |
+-------+--------------+----------+--------------+----------+
5 rows selected (0.433 seconds)


In [ ]:
# Show the employees who earn more than 40000.
%%hive
SELECT * FROM employee WHERE salary > 40000;

+-------+----------+----------+--------------+----------+
|  eid  |   name   |  salary  | designation  |   dept   |
+-------+----------+----------+--------------+----------+
| 1201  | Gopal    | 45000.0  | Technical    | manager  |
| 1202  | Manisha  | 45000.0  | Proof        | reader   |
+-------+----------+----------+--------------+----------+
2 rows selected (1.245 seconds)


In [ ]:
# List the employees sorted by designation.
%%hive
SELECT eid, name, dept, designation FROM employee ORDER BY designation;


+-------+--------------+----------+--------------+
|  eid  |     name     |   dept   | designation  |
+-------+--------------+----------+--------------+
| 1204  | Kiran        | Admin    | Hr           |
| 1205  | Kranthi      | Admin    | Op           |
| 1202  | Manisha      | reader   | Proof        |
| 1203  | Masthanvali  | writer   | Technical    |
| 1201  | Gopal        | manager  | Technical    |
+-------+--------------+----------+--------------+
5 rows selected (5.609 seconds)


In [ ]:
# Count the employees at each salary.
%%hive
SELECT salary, COUNT(*) As counter FROM employee GROUP BY salary;

+----------+----------+
|  salary  | counter  |
+----------+----------+
| 30000.0  | 1        |
| 40000.0  | 2        |
| 45000.0  | 2        |
+----------+----------+
3 rows selected (2.421 seconds)


In [ ]:
# Save the high-salary query as a view called emp_40000.
%%hive
CREATE VIEW emp_40000 AS SELECT * FROM employee WHERE salary > 40000;

No rows affected (0.337 seconds)


In [ ]:
# List the tables and views: emp_40000 is now there.
%%hive
SHOW TABLES;

+------------+
|  tab_name  |
+------------+
| emp_40000  |
| employee   |
+------------+
2 rows selected (0.18 seconds)


In [ ]:
# Query the view like a table.
%%hive
SELECT * FROM emp_40000 LIMIT 5;

+-------+----------+----------+--------------+----------+
|  eid  |   name   |  salary  | designation  |   dept   |
+-------+----------+----------+--------------+----------+
| 1201  | Gopal    | 45000.0  | Technical    | manager  |
| 1202  | Manisha  | 45000.0  | Proof        | reader   |
+-------+----------+----------+--------------+----------+
2 rows selected (0.365 seconds)


In [ ]:
# Remove the view (the employee table is not affected).
%%hive
DROP VIEW emp_40000;

No rows affected (0.568 seconds)


In [ ]:
# Create the Customers table.
%%hive
CREATE TABLE IF NOT EXISTS customers (ID int, Name String, Age int, Address String, Salary Float)
ROW FORMAT DELIMITED FIELDS TERMINATED BY '\t' LINES TERMINATED BY '\n'
STORED AS TEXTFILE;

No rows affected (0.26 seconds)


In [ ]:
# Load Customers.txt into the Customers table.
%%hive
LOAD DATA LOCAL INPATH '/content/Customers.txt' OVERWRITE INTO TABLE customers;

No rows affected (0.699 seconds)


In [ ]:
# Show the customers.
%%hive
SELECT * FROM customers limit 10;

+-----+-----------+------+------------+----------+
| id  |   name    | age  |  address   |  salary  |
+-----+-----------+------+------------+----------+
| 1   | Ramesh    | 32   | Ahmedabad  | 2000.0   |
| 2   | Khilan    | 25   | Delhi      | 1500.0   |
| 3   | kaushik   | 23   | Kota       | 2000.0   |
| 4   | Chaitali  | 25   | Mumbai     | 6500.0   |
| 5   | Hardik    | 27   | Bhopal     | 8500.0   |
| 6   | Komal     | 22   | MP         | 4500.0   |
| 7   | Muffy     | 24   | Indore     | 10000.0  |
+-----+-----------+------+------------+----------+
7 rows selected (0.409 seconds)


In [ ]:
%%hive
CREATE TABLE IF NOT EXISTS Orders (OID int, `Date` String, Customer_ID int, Amount int)
ROW FORMAT DELIMITED FIELDS TERMINATED BY '\t' LINES TERMINATED BY '\n'
STORED AS TEXTFILE;

No rows affected (0.133 seconds)


In [ ]:
# Load Orders.txt into the Orders table.
%%hive
LOAD DATA LOCAL INPATH '/content/Orders.txt' OVERWRITE INTO TABLE Orders;

No rows affected (0.663 seconds)


In [ ]:
# Show the orders.
%%hive
SELECT * FROM Orders;

+------+----------------------+--------------+---------+
| oid  |         date         | customer_id  | amount  |
+------+----------------------+--------------+---------+
| 102  | 2009-10-08 00:00:00  | 3            | 3000    |
| 100  | 2009-10-08 00:00:00  | 3            | 1500    |
| 101  | 2009-11-20 00:00:00  | 2            | 1560    |
| 103  | 2008-05-20 00:00:00  | 4            | 2060    |
+------+----------------------+--------------+---------+
4 rows selected (0.395 seconds)


In [ ]:
# Inner join: customers together with their orders.
%%hive
SELECT c.ID, c.Name, c.Age, o.Amount FROM customers c INNER JOIN Orders o ON (c.ID = o.Customer_ID);

+-----+-----------+------+---------+
| id  |   name    | age  | amount  |
+-----+-----------+------+---------+
| 2   | Khilan    | 25   | 1560    |
| 3   | kaushik   | 23   | 3000    |
| 3   | kaushik   | 23   | 1500    |
| 4   | Chaitali  | 25   | 2060    |
+-----+-----------+------+---------+
4 rows selected (3.233 seconds)


In [ ]:
# Left outer join: every customer, with NULL where there is no order.
%%hive
SELECT c.ID, c.Name, o.Amount, o.`Date` FROM customers c LEFT OUTER JOIN Orders o ON (c.ID = o.Customer_ID);

+-----+-----------+---------+----------------------+
| id  |   name    | amount  |         date         |
+-----+-----------+---------+----------------------+
| 1   | Ramesh    | NULL    | NULL                 |
| 2   | Khilan    | 1560    | 2009-11-20 00:00:00  |
| 3   | kaushik   | 3000    | 2009-10-08 00:00:00  |
| 3   | kaushik   | 1500    | 2009-10-08 00:00:00  |
| 4   | Chaitali  | 2060    | 2008-05-20 00:00:00  |
| 5   | Hardik    | NULL    | NULL                 |
| 6   | Komal     | NULL    | NULL                 |
| 7   | Muffy     | NULL    | NULL                 |
+-----+-----------+---------+----------------------+
8 rows selected (2.07 seconds)


In [ ]:
# Right outer join: every order, with its customer.
%%hive
SELECT c.ID, c.Name, o.Amount, o.`Date` FROM customers c RIGHT OUTER JOIN Orders o ON (c.ID = o.Customer_ID);

+-----+-----------+---------+----------------------+
| id  |   name    | amount  |         date         |
+-----+-----------+---------+----------------------+
| 3   | kaushik   | 3000    | 2009-10-08 00:00:00  |
| 3   | kaushik   | 1500    | 2009-10-08 00:00:00  |
| 2   | Khilan    | 1560    | 2009-11-20 00:00:00  |
| 4   | Chaitali  | 2060    | 2008-05-20 00:00:00  |
+-----+-----------+---------+----------------------+
4 rows selected (2.06 seconds)


In [ ]:
# Full outer join: every row of both tables.
%%hive
SELECT c.ID, c.Name, o.Amount, o.`Date` FROM customers c FULL OUTER JOIN Orders o ON (c.ID = o.Customer_ID);

+-----+-----------+---------+----------------------+
| id  |   name    | amount  |         date         |
+-----+-----------+---------+----------------------+
| 1   | Ramesh    | NULL    | NULL                 |
| 2   | Khilan    | 1560    | 2009-11-20 00:00:00  |
| 3   | kaushik   | 1500    | 2009-10-08 00:00:00  |
| 3   | kaushik   | 3000    | 2009-10-08 00:00:00  |
| 4   | Chaitali  | 2060    | 2008-05-20 00:00:00  |
| 5   | Hardik    | NULL    | NULL                 |
| 6   | Komal     | NULL    | NULL                 |
| 7   | Muffy     | NULL    | NULL                 |
+-----+-----------+---------+----------------------+
8 rows selected (1.828 seconds)


In [ ]:
# List the built-in functions whose name contains "rank".
%%hive
SHOW FUNCTIONS LIKE '%rank%';

+---------------+
|   tab_name    |
+---------------+
| dense_rank    |
| ds_kll_rank   |
| percent_rank  |
| rank          |
+---------------+
4 rows selected (0.083 seconds)


In [ ]:
# Explain what the rank function does.
%%hive
DESCRIBE FUNCTION rank;

+-----------+
| tab_name  |
+-----------+
| rank(x)   |
+-----------+
1 row selected (0.036 seconds)


In [ ]:
# Rank all employees by salary, highest first; ties share a rank and the next rank is skipped.
%%hive
SELECT eid, name, salary, RANK() OVER (ORDER BY salary DESC) AS salary_rank FROM employee;

+-------+--------------+----------+--------------+
|  eid  |     name     |  salary  | salary_rank  |
+-------+--------------+----------+--------------+
| 1202  | Manisha      | 45000.0  | 1            |
| 1201  | Gopal        | 45000.0  | 1            |
| 1204  | Kiran        | 40000.0  | 3            |
| 1203  | Masthanvali  | 40000.0  | 3            |
| 1205  | Kranthi      | 30000.0  | 5            |
+-------+--------------+----------+--------------+
5 rows selected (2.169 seconds)


In [ ]:
# Rank salaries inside each department, without gaps.
%%hive
SELECT dept, salary, DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS salary_rank
FROM employee;

+----------+----------+--------------+
|   dept   |  salary  | salary_rank  |
+----------+----------+--------------+
| Admin    | 40000.0  | 1            |
| Admin    | 30000.0  | 2            |
| manager  | 45000.0  | 1            |
| reader   | 45000.0  | 1            |
| writer   | 40000.0  | 1            |
+----------+----------+--------------+
5 rows selected (2.376 seconds)


In [ ]:
# Number the rows inside each department.
%%hive
SELECT dept, salary, ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS row_no
FROM employee;

+----------+----------+---------+
|   dept   |  salary  | row_no  |
+----------+----------+---------+
| Admin    | 40000.0  | 1       |
| Admin    | 30000.0  | 2       |
| manager  | 45000.0  | 1       |
| reader   | 45000.0  | 1       |
| writer   | 40000.0  | 1       |
+----------+----------+---------+
5 rows selected (1.939 seconds)


In [ ]:
# Share of employees earning the same or less than each employee.
%%hive
SELECT dept, salary, CUME_DIST() OVER (ORDER BY salary) AS cume_dist_value
FROM employee;

+----------+----------+------------------+
|   dept   |  salary  | cume_dist_value  |
+----------+----------+------------------+
| Admin    | 30000.0  | 0.2              |
| Admin    | 40000.0  | 0.6              |
| writer   | 40000.0  | 0.6              |
| reader   | 45000.0  | 1.0              |
| manager  | 45000.0  | 1.0              |
+----------+----------+------------------+
5 rows selected (2.023 seconds)


In [ ]:
# Rank and percent rank of each salary inside its department.
%%hive
SELECT dept, salary, RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS  dept_rank,
Percent_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS percent_rank
FROM employee;

+----------+----------+------------+---------------+
|   dept   |  salary  | dept_rank  | percent_rank  |
+----------+----------+------------+---------------+
| Admin    | 40000.0  | 1          | 0.0           |
| Admin    | 30000.0  | 2          | 1.0           |
| manager  | 45000.0  | 1          | 0.0           |
| reader   | 45000.0  | 1          | 0.0           |
| writer   | 40000.0  | 1          | 0.0           |
+----------+----------+------------+---------------+
5 rows selected (1.99 seconds)


In [ ]:
%%hive
# Split each department's employees into 4 buckets by salary.
SELECT dept, salary, NTILE(4) OVER (PARTITION BY dept ORDER BY salary DESC) AS bucket
FROM employee;

+----------+----------+---------+
|   dept   |  salary  | bucket  |
+----------+----------+---------+
| Admin    | 40000.0  | 1       |
| Admin    | 30000.0  | 2       |
| manager  | 45000.0  | 1       |
| reader   | 45000.0  | 1       |
| writer   | 40000.0  | 1       |
+----------+----------+---------+
5 rows selected (1.733 seconds)


In [ ]:
# Count all rows, and the rows that have a salary.
%%hive
SELECT count(*), count(salary) FROM employee;

+------+------+
| _c0  | _c1  |
+------+------+
| 5    | 5    |
+------+------+
1 row selected (1.956 seconds)


In [ ]:
# Add up all salaries, and each different salary once.
%%hive
SELECT sum(salary), sum(DISTINCT salary) FROM employee;

+-----------+-----------+
|    _c0    |    _c1    |
+-----------+-----------+
| 200000.0  | 115000.0  |
+-----------+-----------+
1 row selected (1.842 seconds)


In [ ]:
# Count the different departments and designations.
%%hive
SELECT count(DISTINCT dept), count(DISTINCT designation) FROM employee;

+------+------+
| _c0  | _c1  |
+------+------+
| 4    | 4    |
+------+------+
1 row selected (1.734 seconds)


In [ ]:
# Names starting with K.
%%hive
SELECT * FROM employee WHERE name LIKE 'K%';

+-------+----------+----------+--------------+--------+
|  eid  |   name   |  salary  | designation  |  dept  |
+-------+----------+----------+--------------+--------+
| 1204  | Kiran    | 40000.0  | Hr           | Admin  |
| 1205  | Kranthi  | 30000.0  | Op           | Admin  |
+-------+----------+----------+--------------+--------+
2 rows selected (0.415 seconds)


In [ ]:
# Names containing m in any case.
%%hive
SELECT * FROM employee WHERE LOWER(name) LIKE '%m%';

+-------+--------------+----------+--------------+---------+
|  eid  |     name     |  salary  | designation  |  dept   |
+-------+--------------+----------+--------------+---------+
| 1202  | Manisha      | 45000.0  | Proof        | reader  |
| 1203  | Masthanvali  | 40000.0  | Technical    | writer  |
+-------+--------------+----------+--------------+---------+
2 rows selected (0.202 seconds)


In [ ]:
# Rows that contain m, or do not: together, every row.
%%hive
SELECT * FROM employee WHERE LOWER(name) LIKE '%m%' OR LOWER(name) NOT LIKE '%m%';

+-------+--------------+----------+--------------+----------+
|  eid  |     name     |  salary  | designation  |   dept   |
+-------+--------------+----------+--------------+----------+
| 1201  | Gopal        | 45000.0  | Technical    | manager  |
| 1202  | Manisha      | 45000.0  | Proof        | reader   |
| 1203  | Masthanvali  | 40000.0  | Technical    | writer   |
| 1204  | Kiran        | 40000.0  | Hr           | Admin    |
| 1205  | Kranthi      | 30000.0  | Op           | Admin    |
+-------+--------------+----------+--------------+----------+
5 rows selected (0.229 seconds)


In [ ]:
# Names matching a regular expression.
%%hive
SELECT * FROM employee WHERE name RLIKE '.*(Gopal|Kiran).*';


+-------+--------+----------+--------------+----------+
|  eid  |  name  |  salary  | designation  |   dept   |
+-------+--------+----------+--------------+----------+
| 1201  | Gopal  | 45000.0  | Technical    | manager  |
| 1204  | Kiran  | 40000.0  | Hr           | Admin    |
+-------+--------+----------+--------------+----------+
2 rows selected (0.232 seconds)


In [ ]:
# Copy the employee table into a new table stored as ORC.
%%hive
CREATE TABLE employee_orc STORED AS ORC AS SELECT * FROM employee;

5 rows affected (1.948 seconds)


In [ ]:
# Show how the new table is defined: note the ORC input and output formats.
%%hive
SHOW CREATE TABLE employee_orc;

+----------------------------------------------------+
|                   createtab_stmt                   |
+----------------------------------------------------+
| CREATE TABLE `employee_orc`(                       |
|   `eid` int,                                       |
|   `name` string,                                   |
|   `salary` float,                                  |
|   `designation` string,                            |
|   `dept` string)                                   |
| ROW FORMAT SERDE                                   |
|   'org.apache.hadoop.hive.ql.io.orc.OrcSerde'      |
| STORED AS INPUTFORMAT                              |
|   'org.apache.hadoop.hive.ql.io.orc.OrcInputFormat'  |
| OUTPUTFORMAT                                       |
|   'org.apache.hadoop.hive.ql.io.orc.OrcOutputFormat' |
| LOCATION                                           |
|   'hdfs://127.0.0.1:9000/user/hive/warehouse/userdb6.db/employee_orc' |
| TBLPROPERTIES (                         

In [ ]:
# Filter the text table: Hive reads and checks every line.
%%hive
SELECT * FROM employee WHERE salary > 40000;

+-------+----------+----------+--------------+----------+
|  eid  |   name   |  salary  | designation  |   dept   |
+-------+----------+----------+--------------+----------+
| 1201  | Gopal    | 45000.0  | Technical    | manager  |
| 1202  | Manisha  | 45000.0  | Proof        | reader   |
+-------+----------+----------+--------------+----------+
2 rows selected (0.358 seconds)


In [ ]:
# Filter the ORC table: Hive can skip blocks using ORC's min/max statistics.
%%hive
SELECT * FROM employee_orc WHERE salary > 40000;

+-------+----------+----------+--------------+----------+
|  eid  |   name   |  salary  | designation  |   dept   |
+-------+----------+----------+--------------+----------+
| 1201  | Gopal    | 45000.0  | Technical    | manager  |
| 1202  | Manisha  | 45000.0  | Proof        | reader   |
+-------+----------+----------+--------------+----------+
2 rows selected (0.428 seconds)


In [ ]:
# List the ORC table's file: a binary file, not readable text like Sample.txt.
!hdfs dfs -ls /user/hive/warehouse/userdb6.db/employee_orc

Found 1 items
-rw-r--r--   1 root supergroup        801 2026-10-26 20:03 /user/hive/warehouse/userdb6.db/employee_orc/000000_0


In [ ]:
# Create a table partitioned by department.
%%hive
CREATE TABLE employee_partitioned (eid int, name String, salary STRING, designation String)
PARTITIONED BY (dept String)
ROW FORMAT DELIMITED FIELDS TERMINATED BY '\t' LINES TERMINATED BY '\n'
STORED AS TEXTFILE;

No rows affected (0.084 seconds)


In [ ]:
# Allow every partition to be created dynamically (for this session).
%%hive
SET hive.exec.dynamic.partition.mode=nonstrict;

No rows affected (0.048 seconds)


In [ ]:
# Fill the partitioned table from employee; Hive creates one partition per department.
%%hive
INSERT OVERWRITE TABLE employee_partitioned PARTITION (dept)
SELECT eid, name, salary, designation, dept FROM employee;

5 rows affected (4.589 seconds)


In [ ]:
# List the partitions Hive created.
%%hive
SHOW PARTITIONS employee_partitioned;

+---------------+
|   partition   |
+---------------+
| dept=Admin    |
| dept=manager  |
| dept=reader   |
| dept=writer   |
+---------------+
4 rows selected (0.076 seconds)


In [ ]:
# List the table's HDFS folder: one sub-folder per department.
!hdfs dfs -ls /user/hive/warehouse/userdb6.db/employee_partitioned

Found 4 items
drwxr-xr-x   - root supergroup          0 2026-10-26 20:08 /user/hive/warehouse/userdb6.db/employee_partitioned/dept=Admin
drwxr-xr-x   - root supergroup          0 2026-10-26 20:08 /user/hive/warehouse/userdb6.db/employee_partitioned/dept=manager
drwxr-xr-x   - root supergroup          0 2026-10-26 20:08 /user/hive/warehouse/userdb6.db/employee_partitioned/dept=reader
drwxr-xr-x   - root supergroup          0 2026-10-26 20:08 /user/hive/warehouse/userdb6.db/employee_partitioned/dept=writer


In [ ]:
# Query one partition: Hive reads only the dept=Admin folder.
%%hive
SELECT * FROM employee_partitioned WHERE dept='Admin';

+-------+----------+----------+--------------+--------+
|  eid  |   name   |  salary  | designation  |  dept  |
+-------+----------+----------+--------------+--------+
| 1205  | Kranthi  | 30000.0  | Op           | Admin  |
| 1204  | Kiran    | 40000.0  | Hr           | Admin  |
+-------+----------+----------+--------------+--------+
2 rows selected (0.589 seconds)


In [ ]:
# Which data does the query read on the partitioned table?
%%hive
EXPLAIN DEPENDENCY SELECT * FROM employee_partitioned WHERE dept='Admin';

+----------------------------------------------------+
|                      Explain                       |
+----------------------------------------------------+
| {"input_tables":[{"tablename":"userdb6@employee_partitioned","tabletype":"MANAGED_TABLE"}],"input_partitions":[{"partitionName":"userdb6@employee_partitioned@dept=Admin"}]} |
+----------------------------------------------------+
1 row selected (0.217 seconds)


In [ ]:
# Which data does the same query read on the unpartitioned table?
%%hive

EXPLAIN DEPENDENCY SELECT * FROM employee WHERE dept='Admin';

+----------------------------------------------------+
|                      Explain                       |
+----------------------------------------------------+
| {"input_tables":[{"tablename":"userdb6@employee","tabletype":"MANAGED_TABLE"}],"input_partitions":[]} |
+----------------------------------------------------+
1 row selected (0.229 seconds)


In [ ]:
# Create a table partitioned by department and bucketed by eid into 4 files.
%%hive
CREATE TABLE employee_bucketed (eid int, name String, salary STRING, designation String)
PARTITIONED BY (dept String)
CLUSTERED BY (eid) INTO 4 BUCKETS
ROW FORMAT DELIMITED FIELDS TERMINATED BY '\t' LINES TERMINATED BY '\n'
STORED AS TEXTFILE;

No rows affected (0.137 seconds)


In [ ]:
# Fill the bucketed table from the partitioned table.
%%hive
INSERT OVERWRITE TABLE employee_bucketed PARTITION (dept)
SELECT eid, name, salary, designation, dept FROM employee_partitioned;

5 rows affected (4.765 seconds)


In [ ]:
# Query the bucketed table.
%%hive
SELECT * FROM employee_bucketed WHERE eid < 1205;

+-------+--------------+----------+--------------+----------+
|  eid  |     name     |  salary  | designation  |   dept   |
+-------+--------------+----------+--------------+----------+
| 1204  | Kiran        | 40000.0  | Hr           | Admin    |
| 1201  | Gopal        | 45000.0  | Technical    | manager  |
| 1202  | Manisha      | 45000.0  | Proof        | reader   |
| 1203  | Masthanvali  | 40000.0  | Technical    | writer   |
+-------+--------------+----------+--------------+----------+
4 rows selected (0.586 seconds)


In [ ]:
# List the bucket files inside the dept=Admin partition.
!hdfs dfs -ls /user/hive/warehouse/userdb6.db/employee_bucketed/dept=Admin

Found 4 items
-rw-r--r--   1 root supergroup          0 2026-10-26 20:14 /user/hive/warehouse/userdb6.db/employee_bucketed/dept=Admin/000000_0
-rw-r--r--   1 root supergroup         22 2026-10-26 20:14 /user/hive/warehouse/userdb6.db/employee_bucketed/dept=Admin/000001_0
-rw-r--r--   1 root supergroup         24 2026-10-26 20:14 /user/hive/warehouse/userdb6.db/employee_bucketed/dept=Admin/000002_0
-rw-r--r--   1 root supergroup          0 2026-10-26 20:14 /user/hive/warehouse/userdb6.db/employee_bucketed/dept=Admin/000003_0


In [ ]:
# Join the bucketed table with the partitioned table on the bucketing column eid.
%%hive
SELECT b.eid, b.name, p.salary FROM employee_bucketed b JOIN employee_partitioned p ON (b.eid = p.eid);

+-------+--------------+----------+
|  eid  |     name     |  salary  |
+-------+--------------+----------+
| 1205  | Kranthi      | 30000.0  |
| 1204  | Kiran        | 40000.0  |
| 1201  | Gopal        | 45000.0  |
| 1202  | Manisha      | 45000.0  |
| 1203  | Masthanvali  | 40000.0  |
+-------+--------------+----------+
5 rows selected (2.851 seconds)


In [ ]:
# Create a table partitioned by department and bucketed by salary into 5 files.
%%hive
CREATE TABLE employee_bucketed_salary (eid INT, name STRING, salary STRING, designation String)
PARTITIONED BY (dept String)
CLUSTERED BY (salary) INTO 5 BUCKETS
ROW FORMAT DELIMITED FIELDS TERMINATED BY '\t' LINES TERMINATED BY '\n'
STORED AS TEXTFILE;

No rows affected (0.141 seconds)


In [ ]:
# Fill the optimised table from employee.
%%hive
INSERT OVERWRITE TABLE employee_bucketed_salary PARTITION (dept)
SELECT eid, name, salary, designation, dept FROM employee;

5 rows affected (3.947 seconds)


In [ ]:
%%hive
# Query the optimised table (employee_bucketed_salary).
SELECT * FROM employee_bucketed_salary WHERE dept = 'Admin';

+-------+----------+----------+--------------+--------+
|  eid  |   name   |  salary  | designation  |  dept  |
+-------+----------+----------+--------------+--------+
| 1205  | Kranthi  | 30000.0  | Op           | Admin  |
| 1204  | Kiran    | 40000.0  | Hr           | Admin  |
+-------+----------+----------+--------------+--------+
2 rows selected (0.397 seconds)


In [ ]:
# @title ▶ 6.1 Install Apache Pig 0.18.0 and turn on %%pig cells
# Pig 0.18 is the first release that runs on Hadoop 3.
PIG_VERSION = '0.18.0'
PIG = LAB / f'pig-{PIG_VERSION}'
PIG_WORK = LAB / 'pig-work'
PIG_WORK.mkdir(exist_ok=True)

# Download, verify and unpack Pig only once per current VM.
if not (PIG / '.verified').exists():
    print(f'Downloading Apache Pig {PIG_VERSION} (about 470 MB) …')
    archive = LAB / 'pig.tar.gz'
    download_apache(f'pig/pig-{PIG_VERSION}/pig-{PIG_VERSION}.tar.gz', archive,
                    ('.sha512', '.sha256', '.sha1', '.md5'), required=False)
    print('Installing Pig …')
    with tarfile.open(archive) as bundle:
        bundle.extractall(LAB, filter='data')
    for script in (PIG / 'bin').iterdir():
        script.chmod(script.stat().st_mode | 0o111)
    (PIG / '.verified').write_text('ok')
    archive.unlink()

# Pig needs Apache Commons Collections 3, which Hadoop 3.5 no longer ships; add it from Maven Central.
COLLECTIONS = PIG / 'lib/commons-collections-3.2.2.jar'
if not COLLECTIONS.exists():
    download_verified('https://repo1.maven.org/maven2/commons-collections/commons-collections/3.2.2/'
                      'commons-collections-3.2.2.jar', COLLECTIONS, ('.sha1',))

# Show only Pig's warnings and errors, so each statement prints just its result.
PIG_LOG4J = PIG_WORK / 'log4j.properties'
PIG_LOG4J.write_text('log4j.rootLogger=WARN, A\n'
                     'log4j.appender.A=org.apache.log4j.ConsoleAppender\n'
                     'log4j.appender.A.target=System.err\n'
                     'log4j.appender.A.layout=org.apache.log4j.PatternLayout\n'
                     'log4j.appender.A.layout.ConversionPattern=%p %c{1}: %m%n\n'
                     'log4j.logger.org.apache.hadoop=ERROR\n'
                     'log4j.logger.org.apache.pig.data.SchemaTupleBackend=ERROR\n')

# Environment for Pig: Java 17 settings, 1 GB of memory, and a plain terminal.
PIG_ENV = {**os.environ, 'PIG_HOME': str(PIG), 'PIG_OPTS': OPENS, 'PIG_HEAPSIZE': '1024', 'TERM': 'dumb',
           'PIG_CLASSPATH': str(COLLECTIONS)}
os.environ['PATH'] = f'{PIG}/bin:' + os.environ['PATH']

# Close a Grunt shell left over from an earlier run of this cell.
if 'PIG_SESSION' in globals() and PIG_SESSION.child is not None and PIG_SESSION.child.isalive():
    PIG_SESSION.child.terminate(force=True)

# One Grunt shell in local mode (-x local reads files from Colab's own disk), kept open for the notebook.
PIG_SESSION = ShellSession(
    [str(PIG / 'bin/pig'), '-x', 'local', '-4', str(PIG_LOG4J)],
    prompt=r'grunt> ', run_command='run {}', echo_command='sh echo {}', env=PIG_ENV, cwd=PIG_WORK)

# Remove full-line -- comments from a cell of Pig Latin.
def pig_latin(cell):
    return '\n'.join(l for l in cell.strip().split('\n') if not l.strip().startswith(('--', '#')))

# Notebook shortcut: a %%pig cell types its Pig Latin into the Grunt shell.
@register_cell_magic
def pig(line, cell):
    text = pig_latin(cell)
    if not text.strip():
        print('(Nothing to run yet: type a Pig Latin statement in this cell, ending with ;)')
        return
    # quit is typed straight into Grunt, as in a terminal, and closes the shell.
    if text.strip().rstrip(';').strip().lower() == 'quit':
        if PIG_SESSION.child is not None and PIG_SESSION.child.isalive():
            PIG_SESSION.child.sendline('quit')
            PIG_SESSION.child.expect(pexpect.EOF, timeout=60)
            PIG_SESSION.child.close()
        PIG_SESSION.child = None
        print('Grunt shell closed. The next %%pig cell will start a new one.')
        return
    # Show results and errors; hide Pig's warnings.
    output = PIG_SESSION.run(text)
    print('\n'.join(l for l in output.split('\n') if not re.match(r'(WARN|INFO) ', l)).strip('\n'))

# Start the Grunt shell now, so the first %%pig cell answers quickly.
if PIG_SESSION.child is None or not PIG_SESSION.child.isalive():
    PIG_SESSION.start()
print('Pig is ready ✓  Grunt shell in local mode · %%pig cells are ready ✓')

  Trying https://dlcdn.apache.org …
  Download: 0%  (1 of 468 MB)
  Download: 10%  (47 of 468 MB)
  Download: 20%  (94 of 468 MB)
  Download: 30%  (141 of 468 MB)
  Download: 40%  (188 of 468 MB)
  Download: 50%  (235 of 468 MB)
  Download: 60%  (282 of 468 MB)
  Download: 70%  (329 of 468 MB)
  Download: 80%  (376 of 468 MB)
  Download: 90%  (423 of 468 MB)
  Download: 100%  (468 of 468 MB)
  MD5 checksum verified: pig.tar.gz
Installing Pig …
  SHA1 checksum verified: commons-collections-3.2.2.jar
Pig is ready ✓  Grunt shell in local mode · %%pig cells are ready ✓


In [ ]:
# Load the text file, then show the new relation's schema and data.
%%pig
pigdata = load '/content/wordcount_simple.txt' as(line:chararray);
describe pigdata;
dump pigdata;

pigdata: {line: chararray}
(Class of CN7031; Apache Pig is a high-level platform for creating programs that run on Apache Hadoop. The language for this platform is called Pig Latin. Pig can execute its Hadoop jobs in MapReduce, Apache Tez, or Apache Spark.)


In [ ]:
# Split every line at spaces into one record per word, then show the result.
%%pig
pigwords = foreach pigdata GENERATE FLATTEN(TOKENIZE(line, ' ')) as word;
describe pigwords;
dump pigwords;

pigwords: {word: chararray}
(Class)
(of)
(CN7031;)
(Apache)
(Pig)
(is)
(a)
(high-level)
(platform)
(for)
(creating)
(programs)
(that)
(run)
(on)
(Apache)
(Hadoop.)
(The)
(language)
(for)
(this)
(platform)
(is)
(called)
(Pig)
(Latin.)
(Pig)
(can)
(execute)
(its)
(Hadoop)
(jobs)
(in)
(MapReduce,)
(Apache)
(Tez,)
(or)
(Apache)
(Spark.)


In [ ]:
# Collect identical words into one group each, then show the groups and their bags.
%%pig
piggrouping = GROUP pigwords by word;
describe piggrouping;
dump piggrouping;


piggrouping: {group: chararray,pigwords: {(word: chararray)}}
(a,{(a)})
(in,{(in)})
(is,{(is),(is)})
(of,{(of)})
(on,{(on)})
(or,{(or)})
(Pig,{(Pig),(Pig),(Pig)})
(The,{(The)})
(can,{(can)})
(for,{(for),(for)})
(its,{(its)})
(run,{(run)})
(Tez,,{(Tez,)})
(jobs,{(jobs)})
(that,{(that)})
(this,{(this)})
(Class,{(Class)})
(Apache,{(Apache),(Apache),(Apache),(Apache)})
(Hadoop,{(Hadoop)})
(Latin.,{(Latin.)})
(Spark.,{(Spark.)})
(called,{(called)})
(CN7031;,{(CN7031;)})
(Hadoop.,{(Hadoop.)})
(execute,{(execute)})
(creating,{(creating)})
(language,{(language)})
(platform,{(platform),(platform)})
(programs,{(programs)})
(MapReduce,,{(MapReduce,)})
(high-level,{(high-level)})


In [ ]:
# Replace each group's bag with the number of words in it, then show the new schema.
%%pig
pig_word_count = FOREACH piggrouping GENERATE group, COUNT(pigwords);
describe pig_word_count;

pig_word_count: {group: chararray,long}


In [ ]:
# Run the steps and print every (word,count) pair.
%%pig
dump pig_word_count;

(a,1)
(in,1)
(is,2)
(of,1)
(on,1)
(or,1)
(Pig,3)
(The,1)
(can,1)
(for,2)
(its,1)
(run,1)
(Tez,,1)
(jobs,1)
(that,1)
(this,1)
(Class,1)
(Apache,4)
(Hadoop,1)
(Latin.,1)
(Spark.,1)
(called,1)
(CN7031;,1)
(Hadoop.,1)
(execute,1)
(creating,1)
(language,1)
(platform,2)
(programs,1)
(MapReduce,,1)
(high-level,1)


In [ ]:
# Leave the Grunt shell.
%%pig
quit;

Grunt shell closed. The next %%pig cell will start a new one.


In [ ]:
# List the files in Pig's working folder: your cell scripts, and a pig_….log file for any error.
!ls -l /content/cn7031-week4-hive/pig-work

total 24
-rw-r--r-- 1 root root  99 Oct 26 20:24 cell_1.txt
-rw-r--r-- 1 root root 108 Oct 26 20:25 cell_2.txt
-rw-r--r-- 1 root root  78 Oct 26 20:26 cell_3.txt
-rw-r--r-- 1 root root  95 Oct 26 20:26 cell_4.txt
-rw-r--r-- 1 root root  21 Oct 26 20:26 cell_5.txt
-rw-r--r-- 1 root root 317 Oct 26 20:22 log4j.properties


In [ ]:

# Delete the userdb6 database with all its tables.
%%hive
DROP DATABASE userdb6 CASCADE;

No rows affected (1.537 seconds)
